In [1]:
import numpy as np
a = np.array([2, 4, 6]); print(a / 2, a // 4)

[1. 2. 3.] [0 1 1]


/ — обычное деление, // — целочисленное

In [ ]:
m = np.ones((3, 4)); print(m.sum(axis=1))

В каждой строке 4 единицы, axis=1 суммирует по строкам

In [ ]:
print(np.arange(3) + np.arange(3).reshape(3, 1))

Строка (3,) и столбец (3,1) растягиваются друг под друга — broadcasting

In [ ]:
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())

sqrt([1,4,9])=[1,2,3], сумма = 6

In [ ]:
print((np.arange(1, 5) ** 2).mean())

[1,2,3,4] в квадрате [1,4,9,16], среднее = 7.5.

In [3]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [4]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [5]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


In [6]:
# 1.
n_clean = clean.sum()
sep_revenue = amount[clean & (month == "2026-09")].sum()
oct_revenue = amount[clean & (month == "2026-10")].sum()
change = (oct_revenue / sep_revenue - 1) * 100
print(n_clean, sep_revenue, oct_revenue, round(change, 1))

4739 35243410.0 34419520.0 -2.3


В чистых данных осталось 4739 заказов (из 5463 всего). Выручка в сентябре — 35 243 410 ₸, в октябре — 34 419 520 ₸, изменение — −2,3%. Это совпадает с ДЗ2, где падение было тоже около −2,3% — то есть без дублей и тестовых заказов реальное падение намного меньше, чем казалось изначально (−22,7%)

In [9]:
# 2. Таблица «категория × месяц»
#Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
#Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?
cats = np.unique(cat)
months = np.array(["2026-09", "2026-10"])
cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                      for c in cats])

for c, r in zip(cats, cat_table):
    print(f"{c:<15} {r[0]:>12,.0f} {r[1]:>12,.0f}")

total_by_cat = cat_table.sum(axis=1)
print("больше всего выручки:", cats[total_by_cat.argmax()])

change_pct = cat_table[:, 1] / cat_table[:, 0] - 1
print("упала сильнее всех:", cats[change_pct.argmin()], round(change_pct.min()*100, 1))


Дом                4,997,730    5,125,790
Книги              1,065,300    1,140,910
Красота            2,610,950    2,678,990
Одежда             9,167,100    8,505,730
Электроника       17,402,330   16,968,100
больше всего выручки: Электроника
упала сильнее всех: Одежда -7.2


Больше всего выручки за два месяца приносит категория Электроника — 34 370 430 ₸. Сильнее всех в процентах упала категория Одежда — минус 7,2% (у остальных категорий либо небольшой рост, либо падение в пределах −2,5%, как у Электроники)

In [10]:
## [База] 3. Средний чек по категориям в октябре
#Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
#Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?
count_table = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                        for c in cats])
avg_check = cat_table / count_table
print(avg_check)
print("самый высокий средний чек (октябрь):", cats[avg_check[:, 1].argmax()])

[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
самый высокий средний чек (октябрь): Электроника


Самый высокий средний чек в октябре — у категории Электроника, 33 600 ₸. Логично, раз это самые дорогие товары из всех пяти категорий

In [11]:
## [Стандарт] 4. Доля городов в выручке месяца
#Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
#Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
share = table / table.sum(axis=0)
print(share)
print(share.sum(axis=0))

[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
[1. 1.]


доля Шымкента в выручке упала с 12,9% в сентябре до 11,7% в октябре — то есть Шымкент стал давать меньшую часть от общей выручки месяца

In [12]:
## [Стандарт] 5. Кто дал падение
#Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
#Напишите менеджеру одну фразу: где искать причину.
total_change = table.sum(axis=0)[1] - table.sum(axis=0)[0]
city_change = table[:, 1] - table[:, 0]
contribution = city_change / total_change
for c, v in zip(cities, contribution):
    print(c, round(v*100, 1), "%")

Актобе 4.4 %
Алматы -17.2 %
Астана 22.3 %
Атырау -5.6 %
Караганда 33.6 %
Шымкент 62.5 %


Шымкент даёт 62,5% от общего падения выручки — это основной источник проблемы. Следом идёт Караганда 33,6% и Астана 22,3%, а Алматы и Атырау на самом деле выросли и немного компенсируют общее падение.

Для менеджера ответ: Основная причина падения выручки в октябре — город Шымкент, на него приходится почти две трети общего снижения.

In [13]:
## [Стандарт] 6. Исправьте код от AI
#AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
#```python
share = table / table.sum(axis=1)

ValueError: operands could not be broadcast together with shapes (6,2) (6,) 

table имеет форму (6, 2) — 6 городов, 2 месяца. table.sum(axis=1) считает сумму по каждой строке и возвращает массив формы (6,) — просто 6 чисел, без информации о столбцах. Когда пытаешься поделить (6,2) на (6,), NumPy не может их совместить (broadcasting не работает, потому что размеры не совпадают и не равны 1) — отсюда ValueError

Фикс: нужно keepdims=True, чтобы сумма осталась в форме (6, 1) вместо (6,) — тогда NumPy сможет растянуть этот единственный столбец на оба месяца:

In [14]:
share = table / table.sum(axis=1, keepdims=True)
print(share)
print(share.sum(axis=1))

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]
[1. 1. 1. 1. 1. 1.]


ошибка в том, что table.sum(axis=1) даёт форму (6,), а не (6,1), и её нельзя разделить на таблицу (6,2) через broadcasting. Добавление keepdims=True сохраняет форму для деления, и тогда получается доля сентября/октября внутри каждого города отдельно